In [1]:
"""
End-to-end compliance check pipeline.
Combine BM25 retrieval + LLM compliance check.

Run: python compliance_pipeline.py
"""

import os
import json
from openai import OpenAI
from dotenv import load_dotenv
from retrieval import CorpusRetriever


load_dotenv('.env.local')

# Setup LLM client
client = OpenAI(
    base_url=os.getenv('GITHUB_MODELS_ENDPOINT'),
    api_key=os.getenv('GITHUB_TOKEN')
)
MODEL = os.getenv('GITHUB_MODELS_DEPLOYMENT')

# Setup retriever
retriever = CorpusRetriever()


def chunk_sop(sop_text: str) -> list:
    """
    Split SOP text into chunks (per paragraph/section).
    Simple implementation: split by double newline.
    """
    chunks = [c.strip() for c in sop_text.split('\n\n') if c.strip()]
    return chunks


def compliance_check_chunk(sop_chunk: str, retrieved_pasal: list) -> dict:
    """
    Run LLM compliance check untuk 1 SOP chunk + retrieved pasal.
    
    Returns:
        Dict with status, gaps, summary
    """
    if not retrieved_pasal:
        return {
            "status": "NEEDS_REVIEW",
            "summary": "Tidak ada pasal regulasi yang cukup relevan dengan section SOP ini.",
            "gaps": []
        }
    
    # Build retrieved context
    retrieved_context = "\n\n".join([
        f"[{r['entry']['sitasi']}] (relevance: {r['score']:.2f})\n{r['entry']['teks']}"
        for r in retrieved_pasal
    ])
    
    prompt = f"""Anda adalah asisten review compliance SOP terhadap regulasi Indonesia.

SOP SECTION:
{sop_chunk}

PASAL REGULASI YANG DI-RETRIEVE:
{retrieved_context}

TUGAS:
Bandingkan SOP section dengan pasal-pasal regulasi.
Identifikasi gap compliance yang KONKRET dan SPESIFIK.

ATURAN KETAT:
1. Hanya gunakan pasal yang SAYA berikan. JANGAN sebut pasal lain.
2. Jika SOP section TIDAK terkait topik pasal, abaikan pasal itu — JANGAN paksa buat gap.
3. Jangan bilang "SOP tidak komprehensif" — sebut spesifik kewajiban apa yang missing.
4. Maksimal 3 gap per chunk.
5. Output HANYA JSON valid, tanpa markdown.

OUTPUT SCHEMA:
{{
  "status": "COMPLIANT" | "GAP_FOUND" | "NEEDS_REVIEW",
  "summary": "1 kalimat ringkasan",
  "gaps": [
    {{
      "sop_excerpt": "kutipan EXACT dari SOP section",
      "regulation_ref": "sitasi exact pasal (misal: Pasal 59 ayat (1) UU 32/2009)",
      "explanation": "kewajiban spesifik yang missing dari SOP, dalam 1-2 kalimat"
    }}
  ]
}}

JSON:"""
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Kamu asisten compliance review. Output JSON valid saja."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=800,
        temperature=0.2,
        response_format={"type": "json_object"}
    )
    
    raw = response.choices[0].message.content
    
    try:
        result = json.loads(raw)
        result['_tokens'] = response.usage.total_tokens
        return result
    except json.JSONDecodeError as e:
        return {
            "status": "NEEDS_REVIEW",
            "summary": f"JSON parse error: {str(e)[:100]}",
            "gaps": [],
            "_raw": raw,
            "_tokens": response.usage.total_tokens
        }


def validate_citations(gaps: list, retrieved_pasal: list) -> list:
    """
    Anti-hallucination guardrail:
    Drop gaps yang regulation_ref-nya tidak ada di retrieved pasal.
    """
    valid_sitasi = {r['entry']['sitasi'] for r in retrieved_pasal}
    valid_gaps = []
    dropped = 0
    
    for gap in gaps:
        ref = gap.get('regulation_ref', '')
        if ref in valid_sitasi:
            valid_gaps.append(gap)
        else:
            dropped += 1
            print(f"  [Guardrail] Dropped gap with invalid citation: {ref}")
    
    if dropped > 0:
        print(f"  [Guardrail] Dropped {dropped} gap(s) due to citation validation")
    
    return valid_gaps


def analyze_sop(sop_text: str, top_k: int = 5) -> dict:
    """
    Full pipeline: SOP text in -> compliance analysis JSON out.
    """
    print(f"\n{'='*60}")
    print("ANALYZING SOP")
    print('='*60)
    
    # Step 1: Chunk SOP
    chunks = chunk_sop(sop_text)
    print(f"SOP split into {len(chunks)} chunk(s)")
    
    all_gaps = []
    chunk_results = []
    total_tokens = 0
    
    # Step 2: Process each chunk
    for i, chunk in enumerate(chunks, 1):
        print(f"\n--- Chunk {i}/{len(chunks)} ---")
        print(f"Text: {chunk[:100]}...")
        
        # Retrieve relevant pasal
        retrieved = retriever.retrieve(chunk, top_k=top_k, min_score=0.5)
        print(f"Retrieved {len(retrieved)} pasal:")
        for r in retrieved:
            print(f"  - {r['entry']['sitasi']} (score: {r['score']:.2f})")
        
        # LLM compliance check
        if retrieved:
            result = compliance_check_chunk(chunk, retrieved)
            
            # Apply citation validation guardrail
            if result.get('gaps'):
                result['gaps'] = validate_citations(result['gaps'], retrieved)
            
            print(f"Status: {result.get('status')}, Gaps: {len(result.get('gaps', []))}")
            chunk_results.append(result)
            all_gaps.extend(result.get('gaps', []))
            total_tokens += result.get('_tokens', 0)
        else:
            print("Skipped (no relevant pasal retrieved)")
            chunk_results.append({
                "status": "NEEDS_REVIEW",
                "summary": "No relevant regulation found for this section.",
                "gaps": []
            })
    
    # Step 3: Aggregate result
    print(f"\n{'='*60}")
    print("AGGREGATING FINAL RESULT")
    print('='*60)
    
    # Cap gaps at 5 max
    all_gaps = all_gaps[:5]
    
    # Determine overall status
    if any(c.get('status') == 'GAP_FOUND' for c in chunk_results):
        overall_status = 'GAP_FOUND' if all_gaps else 'NEEDS_REVIEW'
    elif all(c.get('status') == 'COMPLIANT' for c in chunk_results):
        overall_status = 'COMPLIANT'
    else:
        overall_status = 'NEEDS_REVIEW'
    
    # Build summary
    if overall_status == 'COMPLIANT':
        summary = "SOP appears compliant with retrieved regulations."
    elif overall_status == 'GAP_FOUND':
        summary = f"Found {len(all_gaps)} compliance gap(s) against UU 32/2009."
    else:
        summary = "Insufficient evidence for confident assessment. Manual review recommended."
    
    final = {
        "status": overall_status,
        "summary": summary,
        "primary_regulation": "UU 32/2009 PPLH",
        "gaps": all_gaps,
        "disclaimer": "Pre-check otomatis berbasis pasal terpilih. Bukan opini hukum final; gunakan untuk review awal internal.",
        "_meta": {
            "chunks_processed": len(chunks),
            "total_tokens": total_tokens
        }
    }
    
    return final


# Self-test
if __name__ == '__main__':
    # SOP dummy untuk test
    sop_test = """SOP Penanganan Limbah PT Contoh
    
Bagian 1: Limbah dari proses produksi disimpan di area khusus pabrik.

Bagian 2: Karyawan wajib memakai APD saat menangani limbah.

Bagian 3: Pengangkutan limbah dilakukan setiap minggu oleh pihak ketiga."""
    
    result = analyze_sop(sop_test)
    
    print(f"\n{'='*60}")
    print("FINAL RESULT")
    print('='*60)
    print(json.dumps(result, indent=2, ensure_ascii=False))

ModuleNotFoundError: No module named 'retrieval'